In [1]:
#Apertura del Navegador e Intervención Inicial
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time

# --- CONFIGURACIÓN DEL ENTORNO VISUAL ---
options = Options()
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
# No usamos headless para que el estudiante vea la GUI en el contenedor
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

# Iniciar el driver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Navegar a Pisos.com
print(" Abriendo navegador en el contenedor...")
driver.get("https://www.pisos.com/venta/pisos-palencia/")

# --- PAUSA DE INTERVENCIÓN HUMANA ---
print("\n ACCIÓN REQUERIDA:")
print("1. Ve a la pestaña del escritorio del contenedor (VNC).")
print("2. Verifica si aparece un Captcha y resuélvelo manualmente.")
print("3. Una vez que veas el listado de laptops, regresa aquí.")

input("\n➔ Presiona ENTER en esta celda cuando estés listo para continuar...")

# --- VERIFICACIÓN DE BLOQUEO (Capítulo 7.4) ---
html_check = driver.page_source.lower()
if "captcha" in html_check or "robot" in html_check:
    print("ADVERTENCIA: Aún se detecta un bloqueo en el HTML.")
else:
    print("Confirmado: Navegador listo y sin bloqueos evidentes.")

 Abriendo navegador en el contenedor...

 ACCIÓN REQUERIDA:
1. Ve a la pestaña del escritorio del contenedor (VNC).
2. Verifica si aparece un Captcha y resuélvelo manualmente.
3. Una vez que veas el listado de laptops, regresa aquí.



➔ Presiona ENTER en esta celda cuando estés listo para continuar... 


ADVERTENCIA: Aún se detecta un bloqueo en el HTML.


In [2]:
#Configuración de Persistencia (MongoDB)
from pymongo import MongoClient

# --- CONEXIÓN A MONGODB ---
try:
    # 1. Conectar al contenedor de Mongo
    cliente = MongoClient('mongodb://database:27017/')
    db = cliente['pisos_db_']
    coleccion = db['palencia_data']
    
    # Limpieza inicial de la colección para el ejercicio
    coleccion.delete_many({})
    print("Conexión a MongoDB exitosa. Base de datos preparada.")
except Exception as e:
    print(f"Error al conectar con MongoDB: {e}")

Conexión a MongoDB exitosa. Base de datos preparada.


In [3]:
#Scraping de Página 1 y Salto a Página 2
def extraer_inmobiliarios():
    """Función para capturar datos de inmuebles"""
    propiedades = driver.find_elements(By.CSS_SELECTOR, '.ad-preview__info')
    lista = []
    
    for p in propiedades:
        try:
            titulo = p.find_element(By.CSS_SELECTOR, '.ad-preview__title').text 
            precio = p.find_element(By.CSS_SELECTOR, '.ad-preview__price').text 
            
            try:
                caracteristicas = p.find_element(By.CSS_SELECTOR, '.ad-preview__char').text
            except:
                caracteristicas = "No disponible"
            
            lista.append({
                "propiedad": titulo,
                "precio": precio,
                "habitaciones": caracteristicas,
                "fecha": time.strftime("%Y-%m-%d %H:%M")
            })
        except:
            continue
            
    return lista

# --- PROCESAR PÁGINA 1 ---
print("Extrayendo datos de la Página 1...")
datos_p1 = extraer_inmobiliarios()
if datos_p1:
    coleccion.insert_many(datos_p1)
    print(f" Guardados {len(datos_p1)} productos de la Página 1.")

# --- PAUSA PARA CAMBIO DE PÁGINA ---
print("\n SEGUNDA INTERVENCIÓN:")
print("1. Ve al navegador y haz clic en 'Siguiente' (Next) al final de la página.")
print("2. Espera a que carguen los nuevos resultados.")
input("\n➔ Presiona ENTER aquí cuando la Página 2 haya cargado...")

# Verificación de seguridad en el cambio de página
if "captcha" in driver.page_source.lower():
    print(" ¡Atención! Apareció un Captcha al cambiar de página. Resuélvelo en la GUI.")
    input("Presiona ENTER una vez resuelto...")

# --- PROCESAR PÁGINA 2 ---
print("Extrayendo datos de la Página 2...")
datos_p2 = extraer_inmobiliarios()
if datos_p2:
    coleccion.insert_many(datos_p2)
    print(f" Guardados {len(datos_p2)} productos de la Página 2.")

Extrayendo datos de la Página 1...
 Guardados 30 productos de la Página 1.

 SEGUNDA INTERVENCIÓN:
1. Ve al navegador y haz clic en 'Siguiente' (Next) al final de la página.
2. Espera a que carguen los nuevos resultados.



➔ Presiona ENTER aquí cuando la Página 2 haya cargado... 


Extrayendo datos de la Página 2...
 Guardados 30 productos de la Página 2.


In [4]:
# --- VALIDACIÓN FINAL ---
total_docs = coleccion.count_documents({})
print(f"\n RESUMEN FINAL:")
print(f"Total de registros en MongoDB: {total_docs}")

if total_docs >= 3:
    print(" REQUISITO CUMPLIDO: Se capturaron los datos exitosamente.")
else:
    print(" REQUISITO NO CUMPLIDO: Revisa los selectores o el bloqueo del sitio.")

# Finalizar sesión
driver.quit()
print("Navegador cerrado.")


 RESUMEN FINAL:
Total de registros en MongoDB: 60
 REQUISITO CUMPLIDO: Se capturaron los datos exitosamente.
Navegador cerrado.
